In [ ]:
import os
import pandas as pd

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")
RESULTS_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "results")
OUTPUT_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "output")
ENSEMBLE_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "ensemble")
MODELS_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "models")
TRAINING_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "training")

# Make sure the output directories exist
os.makedirs(ENSEMBLE_DIR_PATH, exist_ok=True)
os.makedirs(MODELS_DIR_PATH, exist_ok=True)
os.makedirs(TRAINING_DIR_PATH, exist_ok=True)

## Load data

In [ ]:
training_df_log_transformed = pd.read_csv(
    os.path.join(TRAINING_DIR_PATH, "training_df_log_transformed.csv"))

training_df_log_transformed.head()

## Create Projections

In [ ]:
from utils.ml_utils_v2 import EnsembleProjections
ep = EnsembleProjections()

In [ ]:
numeric_cols = training_df_log_transformed.select_dtypes(include=["float64", "int64"]).columns.tolist()
numeric_cols_to_drop = ["year", "log_total_emissions"]
numeric_cols = [col for col in numeric_cols if col not in numeric_cols_to_drop]
numeric_cols

In [ ]:
# Arima params:
n_scenarios = 10

In [ ]:
ensemble_arima_df = ep.generate_ensemble_arima(
    df=training_df_log_transformed,
    feature_cols=numeric_cols,
    start_year=2022,
    end_year=2030,
    n_scenarios=n_scenarios,
    arima_order=(1,1,1),
    random_state=42
)


In [ ]:
# If you don't want to generate auto-tuned ARIMA models, you can comment this out
ensemble_arima_tuned_df = ep.generate_ensemble_arima(
    df=training_df_log_transformed,
    feature_cols=numeric_cols,
    start_year=2022,
    end_year=2030,
    n_scenarios=n_scenarios,
    auto_tune_arima=True,
    random_state=42
)

In [ ]:
# Make sure ensemble_arima_df is sorted by iso_alpha_3 and year
ensemble_arima_df = ensemble_arima_df.sort_values(by=["iso_alpha_3", "year"]).reset_index(drop=True)
ensemble_arima_df.head()

In [ ]:
ensemble_arima_tuned_df = ensemble_arima_tuned_df.sort_values(by=["iso_alpha_3", "year"]).reset_index(drop=True)
ensemble_arima_tuned_df.head()

In [ ]:
# Save the ensemble dataframes to csv files
ensemble_arima_df.to_csv(os.path.join(ENSEMBLE_DIR_PATH, f"ensemble_arima_{n_scenarios}_df.csv"), index=False)
ensemble_arima_tuned_df.to_csv(os.path.join(ENSEMBLE_DIR_PATH, f"ensemble_arima_tuned_{n_scenarios}_df.csv"), index=False)  # Comment this line if you don't want to generate auto-tuned ARIMA models